# scarlatti-doodle project!

## Zip 파일 메모리에 로딩
#### this code block was written using AI

In [19]:
import io
import os
import zipfile
import tempfile
import partitura as pt
from music21 import midi, environment

# music21 임시 디렉토리 억까 방지 설정
try:
    environment.Environment()['directoryScratch'] = '/tmp'
except:
    pass

def mount_scarlatti():
    """scarlatti.zip을 가상 메모리에 마운트하고, 파일 목록을 반환합니다."""
    ZIP_FILE_PATH = "original_midi.zip"
    
    if not os.path.exists(ZIP_FILE_PATH):
        raise FileNotFoundError(f"⚠️ '{ZIP_FILE_PATH}' 파일이 없습니다. 왼쪽 탐색기에 업로드해 주세요!")
        
    print("🛸 가상 메모리(RAM)에 스칼라티 555개 소나타 마운트 중...")
    
    with open(ZIP_FILE_PATH, "rb") as f:
        zip_buffer = io.BytesIO(f.read())
        
    archive = zipfile.ZipFile(zip_buffer)
    midi_files = [f for f in archive.namelist() if f.lower().endswith(('.mid', '.midi'))]
    print(f"📦 마운트 완료! 총 {len(midi_files)}개의 가상 미디 파일 준비 완료.")
    
    return archive, midi_files

def load_midi_from_virtual_folder(archive, file_path, target_type="partitura"):
    """
    [에러 수정 완료] 가상 폴더에서 데이터를 읽어 지정한 라이브러리 객체로 변환합니다.
    BytesIO 타입 제한 억까를 우회하기 위해 OS 임시 파일 핸들러를 안전하게 사용합니다.
    """
    # 1. 압축 파일 내부에서 순수 바이트 데이터 추출
    midi_raw_bytes = archive.read(file_path)
    
    # 2. 파르티투라 객체로 변환할 때
    if target_type == "partitura":
        # 💡 [핵심 우회] BytesIO를 거부하므로, RAM처럼 작동하는 임시 파일을 시스템에 잠깐 썼다 지웁니다.
        # 디스크에 영구 저장되지 않고 함수가 끝나면 메모리에서 자동 소멸해서 속도가 엄청 빠릅니다!
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mid") as tmp_file:
            tmp_file.write(midi_raw_bytes)
            tmp_file_path = tmp_file.name
        
        try:
            # 안전하게 문자열 경로(PathLike)로 인식시켜서 에러 타파!
            performance = pt.load_performance_midi(tmp_file_path)[0]
        finally:
            # 처리가 끝나면 임시 파일 흔적 지우기
            if os.path.exists(tmp_file_path):
                os.remove(tmp_file_path)
                
        return performance
        
    # 3. 뮤직21 객체로 변환할 때 (기존과 동일, 잘 작동함)
    elif target_type == "music21":
        mf = midi.MidiFile()
        mf.readstr(midi_raw_bytes)
        m21_score = midi.translate.midiFileToStream(mf)
        return m21_score
        
    else:
        raise ValueError("⚠️ target_type은 'partitura' 또는 'music21'만 가능합니다!")

# 🔥 [실행] 가상 폴더 연결하기
scarlatti_folder, file_list = mount_scarlatti()
print("앞으로 scarlatti_folder, file_list에 접근해서 사용!")

🛸 가상 메모리(RAM)에 스칼라티 555개 소나타 마운트 중...
📦 마운트 완료! 총 555개의 가상 미디 파일 준비 완료.
앞으로 scarlatti_folder, file_list에 접근해서 사용!


#### 전처리 함수들 준비

In [1]:
from music21 import *
import partitura as pt

# 흐름
# 미디파일 -> 모든조로 전조 -> 렌덤으로 마디 분리 -> 오른손 왼손 분리 -> 오른손 데이터는 마스킹 -> 마스킹한 오른손 데이터는 인풋으로, 온전한 오른손과 왼손 데이터는 아웃풋으로 넘파이 어레이로 전환해서 ai모델 학습용 리스트에 저장

def Score_TrebleBassSeparation_Partitura(partituraPerformance: pt.performance.PerformedPart, splitPoint: int = 60):
    '''높은음자리표 낮은음자리표 분리해서 리턴하는 함수'''
    performance = partituraPerformance[0]
    note_array = performance.note_array()
    
    split_pitch = splitPoint
    
    treble_mask = note_array['pitch'] >= split_pitch
    bass_mask = note_array['pitch'] < split_pitch
    
    treble_note_array = note_array[treble_mask] # 이거는 단순 마스크! 프린트하면 true와 false 로 이루어진 어레이가 나옴!
    bass_note_array = note_array[bass_mask]
    
    # NumPy 배열을 partitura가 저장할 수 있는 PerformedPart 객체 상자에 다시 담아줍니다.
    treble_part = pt.performance.PerformedPart.from_note_array(treble_note_array)
    bass_part = pt.performance.PerformedPart.from_note_array(bass_note_array)
    
    return treble_part, bass_part


def Score_TransposeToAllKeys(partituraPerformance : pt.performance.PerformedPart, music21Score : stream.Score):
    '''스코어 파일을 모든 조로 전조해서 리턴'''
    transposedPerformances = []
    
    KEY_TO_NUM = {
    'C': 0, 'C#': 1, 'D': 2, 'D#': 3, 'E': 4, 'F': 5,
    'F#': 6, 'G': 7, 'G#': 8, 'A': 9, 'A#': 10, 'B': 11,
    'Db': 1, 'Eb': 3, 'Gb': 6, 'Ab': 8, 'Bb': 10
    }

    predicted_key = str(music21Score.analyze('key').tonic.name)
    num = KEY_TO_NUM[predicted_key.replace('-', 'b').upper()]

    note_array = partituraPerformance[0].note_array().copy()
    note_array["pitch"] -= num

    for _ in range(0, 12):
        transposedPerformances.append(pt.performance.PerformedPart.from_note_array(note_array))
        note_array["pitch"] += 1


    return transposedPerformances

def Score_SliceByMeasures(inputScore):
    '''스코어 파일을 특정 마디 길이만큼 나눠서 리턴'''
    # 미디를 16마디(32마디로 할까?)로 나눔, 0에서 2 사이의 마디만큼 겹침, 남은 마디가 부족하면 겹쳐서라도 16마디 맟춤

    pass


def Score_MaskNotes(inputScore):
    '''멜로디 데이터에서 일부 데이터들 마스킹해서 리턴'''
    pass

def ScoreToDataset(inputScore):
    '''스코어 파일을 ai 학습용 데이터셋으로 변환해서 리턴, 코드 정보와 박자정보 추가'''
    # 코드랑 박자정보 추가는 할지 말지 모르겠다...
    pass

    


/home/codespace/.local/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


In [2]:
from music21 import *

# 1. 함수 실행 및 완벽히 정제된 NumPy 배열 2개 받아오기
partituraTestFile = pt.load_performance_midi("sonatas_k-531_(c)sankey.mid")
music21TestFile = converter.parse("sonatas_k-531_(c)sankey.mid")


treble_data, bass_data = Score_TrebleBassSeparation_Partitura(
    partituraPerformance = partituraTestFile
)

# 2. 결과물 확인을 위해 각각 개별 미디 파일로 안전하게 저장하기
pt.save_performance_midi(treble_data, "partitura_treble_output.mid")
pt.save_performance_midi(bass_data, "partitura_bass_output.mid")

print("✨ 파르티투라 분리 완료! 박자가 완벽하게 고정된 미디 파일이 생성되었습니다.")

transposedArray = Score_TransposeToAllKeys(partituraPerformance= partituraTestFile, music21Score= music21TestFile)

pt.save_performance_midi(transposedArray[0], "transposedtoC.mid")



✨ 파르티투라 분리 완료! 박자가 완벽하게 고정된 미디 파일이 생성되었습니다.
